<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_4_PDE_Experiment_in_1D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
GENERIC-FNO Benchmark
=====================
Fourier Neural Operator with GENERIC thermodynamic structure.

Architecture:
  - E-net (FNO → scalar): learns energy functional E[u]
  - S-net (FNO → scalar): learns entropy functional S[u]
  - L(k): anti-Hermitian diagonal operator (reversible dynamics)
  - M(k): Hermitian PSD diagonal operator (dissipative dynamics)
  - Dynamics: du/dt = L·δE/δu + M·δS/δu
  - Hard projection: energy conservation + entropy non-decrease

Compares: FNO (vanilla), EP-FNO (energy penalty), GENERIC-FNO (ours)
Tests on: Heat, Wave, Burgers, Poisson, Biharmonic

Key insight: dynamics are CONSTRUCTED from E,S — no bypass possible.
Unlike prior FMO approach where physics was a soft auxiliary loss on an
unconstrained backbone, here the model MUST learn meaningful E and S
to produce correct predictions.

Reference gap: No prior work embeds full GENERIC (energy conservation +
entropy production) into function-space neural operators (FNO/DeepONet).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict

# ============================================================================
# Data Generation (same 5 PDEs)
# ============================================================================

def generate_heat_data(n_samples=200, nx=64, nt=20, dt=0.01, nu=0.01):
    """Heat equation: du/dt = nu * d²u/dx². Purely dissipative."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    decay = torch.exp(-nu * k**2 * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)  # (nt+1, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'heat'

def generate_wave_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0):
    """Wave equation (1st order system): du/dt = c*dv/dx, dv/dt = c*du/dx.
    Purely reversible (Hamiltonian). We track u-component only."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    omega = c * k  # dispersion relation

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        v0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)
            # Give some initial velocity too
            amp_v = torch.randn(1).item() * 0.3
            v0 += amp_v * torch.cos(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        v_hat = torch.fft.rfft(v0)

        traj = [u0.clone()]
        for t in range(nt):
            # Exact solution: rotate in (u_hat, v_hat) space
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            u_new = cos_w * u_hat + 1j * sin_w * v_hat
            v_new = 1j * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'wave'

def generate_advection_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0, max_mode=6):
    """1D linear advection: du/dt + c*du/dx = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly. Clean fully-observed reversible scalar test
    => a thermodynamically-consistent operator should drive M -> 0. Band-limited
    to max_mode so content stays within the operator range (no high-k aliasing)."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    phase = torch.exp(-1j * c * k * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, max_mode+1, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            ph = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + ph)
        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft(u_hat, n=nx))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'

def generate_burgers_data(n_samples=200, nx=64, nt=20, dt=0.005, nu=0.02):
    """Viscous Burgers: du/dt + u*du/dx = nu*d²u/dx². Mixed rev+diss."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    dx = x[1] - x[0]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, 5, (1,)).item()
            amp = torch.randn(1).item() * 0.3
            phase = torch.rand(1).item() * 2 * math.pi
            u += amp * torch.sin(ki * x + phase)

        traj = [u.clone()]
        # Semi-implicit: diffusion in spectral, advection in physical
        for t in range(nt):
            u_hat = torch.fft.rfft(u)
            # Diffusion (implicit)
            u_hat = u_hat / (1 + nu * k**2 * dt)
            u = torch.fft.irfft(u_hat, n=nx)
            # Advection (explicit, spectral derivative)
            du_dx = torch.fft.irfft(1j * k * torch.fft.rfft(u), n=nx)
            u = u - dt * u * du_dx
            traj.append(u.clone())

        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'burgers'

# ============================================================================
# Building Blocks
# ============================================================================

class SpectralConv1d(nn.Module):
    """Standard FNO spectral convolution (1D). Full-rank."""
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.modes = modes
        scale = 1.0 / (in_ch * out_ch)
        self.W = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes, dtype=torch.cfloat))

    def forward(self, x):
        # x: (B, C, N)
        B, C, N = x.shape
        x_hat = torch.fft.rfft(x, dim=-1)
        m = min(self.modes, x_hat.shape[-1])
        out_hat = torch.zeros(B, self.W.shape[0], x_hat.shape[-1],
                              dtype=torch.cfloat, device=x.device)
        out_hat[:, :, :m] = torch.einsum('bix,oix->box', x_hat[:, :, :m], self.W[:, :, :m])
        return torch.fft.irfft(out_hat, n=N)


class FNO_Block(nn.Module):
    """Single FNO layer: spectral conv + skip + activation."""
    def __init__(self, width, modes):
        super().__init__()
        self.conv = SpectralConv1d(width, width, modes)
        self.skip = nn.Conv1d(width, width, 1)
        self.norm = nn.InstanceNorm1d(width)

    def forward(self, x):
        return F.gelu(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone(nn.Module):
    """Multi-layer FNO backbone: lift → N layers → output channels."""
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4):
        super().__init__()
        self.lift = nn.Conv1d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block(width, modes) for _ in range(n_layers)])
        self.proj = nn.Sequential(
            nn.Conv1d(width, width, 1),
            nn.GELU(),
            nn.Conv1d(width, out_ch, 1)
        )

    def forward(self, x):
        # x: (B, in_ch, N)
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet(nn.Module):
    """FNO backbone → scalar functional F[u].
    Maps field u(x) to a single scalar per sample.
    Uses FNO layers to process the field, then global integration."""

    def __init__(self, width=24, modes=12, n_layers=3):
        super().__init__()
        self.backbone = FNO_Backbone(in_ch=1, out_ch=1, width=width,
                                      modes=modes, n_layers=n_layers)
        # After backbone: (B, 1, N) → density field
        # Integrate over space to get scalar: F = ∫f(x)dx ≈ mean(f) * L
        # Then pass through small MLP for flexibility
        self.head = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, u):
        """u: (B, 1, N) → F: (B,)"""
        density = self.backbone(u)           # (B, 1, N) — energy/entropy density
        integral = density.mean(dim=-1)      # (B, 1) — spatial average ∝ integral
        return self.head(integral).squeeze(-1)  # (B,)


# ============================================================================
# Model 1: Vanilla FNO (baseline)
# ============================================================================

class VanillaFNO(nn.Module):
    """Standard Fourier Neural Operator with residual skip.
    Predicts increment: u_next = u + FNO(u).
    Fair comparison to GENERIC-FNO which also uses u + increment."""
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone(in_ch=1, out_ch=1, width=width,
                                      modes=modes, n_layers=n_layers)

    def forward(self, u):
        """u: (B, 1, N) → u_next: (B, 1, N)"""
        return u + self.backbone(u)

    def predict_with_info(self, u):
        """Return prediction + empty info dict for uniform interface."""
        u_next = self.forward(u)
        return u_next, {}


# ============================================================================
# Model 2: EP-FNO (Energy-Penalized FNO)
# ============================================================================

class EP_FNO(nn.Module):
    """FNO + energy penalty loss. Soft physics constraint.
    Residual: u_next = u + FNO(u)."""
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone(in_ch=1, out_ch=1, width=width,
                                      modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        # Compute energy for penalty
        E_in = 0.5 * (u**2).mean(dim=-1).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=-1).mean(dim=-1)
        dE = E_out - E_in
        return u_next, {'energy_in': E_in, 'energy_out': E_out, 'dE': dE}

    def energy_penalty(self, info, pde_type):
        """One-sided penalty: penalize energy increase for dissipative PDEs."""
        dE = info['dE']
        if pde_type in ('heat', 'burgers'):
            return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'):
            return (dE**2).mean()  # energy should be conserved
        else:
            return torch.tensor(0.0, device=dE.device)


# ============================================================================
# Model 3: GENERIC-FNO (our method)
# ============================================================================

class GENERIC_FNO(nn.Module):
    """
    Dynamics constructed from GENERIC structure:
        du/dt = L · δE/δu + M · δS/δu

    With hard projection:
        1. Energy conservation: du/dt projected ⊥ δE/δu
        2. Entropy non-decrease: dS/dt ≥ 0 enforced

    E-net and S-net are FNO backbones → scalar functionals.
    L(k) is anti-Hermitian diagonal (reversible, per Fourier mode).
    M(k) is Hermitian PSD diagonal (dissipative, per Fourier mode).
    """

    def __init__(self, nx=64, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, degeneracy_construction=True, l2_vargrad=False,
                 use_residual=True):
        super().__init__()
        self.nx = nx
        self.modes_op = modes_op
        # degeneracy_construction (DEFAULT): build L=(I-P_S)D_L(I-P_S),
        #   M=(I-P_E)D_M(I-P_E) so L dS=0 and M dE=0 EXACTLY => energy conserved and
        #   entropy produced by construction (no projection, no correction, no
        #   residual). Reversible PDEs are forced to learn M->0. Set False for the
        #   legacy projection+correction path (ablation; does not specialize).
        # l2_vargrad: scale dE,dS by N/|Omega| (L2 variational derivative); affects
        #   only operator-output scale, default off. use_residual: legacy path only.
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad
        self.use_residual = use_residual
        n_rfft = nx // 2 + 1
        m = min(modes_op, n_rfft)
        self.m = m

        # Functional networks
        self.E_net = FunctionalNet(width=width_func, modes=modes_func,
                                    n_layers=n_layers_func)
        self.S_net = FunctionalNet(width=width_func, modes=modes_func,
                                    n_layers=n_layers_func)

        # L operator: anti-Hermitian diagonal in Fourier space
        # L(k) = i * a(k), where a(k) is real → purely imaginary multiplier
        # Anti-symmetry: ⟨f, Lf⟩ = 0 for real fields (automatic)
        # NOTE: L and M absorb the time step dt — no separate dt parameter.
        # Init scale ~0.3 so initial increment is O(0.01-0.1), giving gradient signal.
        self.a = nn.Parameter(0.3 * torch.randn(m))

        # M operator: Hermitian PSD diagonal in Fourier space
        # M(k) = |b(k)|² ≥ 0 → real non-negative multiplier
        self.b_real = nn.Parameter(0.3 * torch.randn(m))
        self.b_imag = nn.Parameter(0.3 * torch.randn(m))

        # Small residual path for modes beyond the GENERIC operators
        # (handles high-frequency content not captured by truncated L/M)
        self.residual = nn.Sequential(
            nn.Conv1d(1, 16, 1),
            nn.GELU(),
            nn.Conv1d(16, 1, 1)
        )
        self.residual_gate = nn.Parameter(torch.tensor(-3.0))  # starts near 0

    def _get_operators(self):
        """Return L(k) and M(k) with correct structure."""
        # L(k) = i * a(k) — purely imaginary, anti-Hermitian
        L_k = 1j * self.a  # (modes,) complex

        # M(k) = |b(k)|² — real, non-negative (PSD)
        M_k = self.b_real**2 + self.b_imag**2  # (modes,) real

        return L_k, M_k

    # --- single-operator Fourier multipliers (degeneracy-by-construction) ---
    def _L_apply(self, v):
        """Apply skew diagonal D_L = i*a to physical 1D field v (B,1,N)."""
        N = v.shape[-1]; m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        out[:, :, :m] = (1j * self.a) * vh[:, :, :m]
        return torch.fft.irfft(out, n=N)

    def _M_apply(self, v):
        """Apply PSD diagonal D_M = |b|^2 to physical 1D field v (B,1,N)."""
        N = v.shape[-1]; m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        M_k = self.b_real**2 + self.b_imag**2
        out[:, :, :m] = M_k * vh[:, :, :m]
        return torch.fft.irfft(out, n=N)

    @staticmethod
    def _remove(v, w):
        """(I - P_w) v: remove component of v along direction w, per sample."""
        ip = (v * w).sum(dim=-1, keepdim=True)
        nn_ = (w * w).sum(dim=-1, keepdim=True) + 1e-12
        return v - (ip / nn_) * w

    def _generic_rhs(self, dEdu, dSdu):
        """du/dt = (I-P_S)D_L(I-P_S)dE + (I-P_E)D_M(I-P_E)dS. Degeneracy exact =>
        dE/dt=0 and dS/dt=<dS,M dS>>=0 by construction, no projection needed."""
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu)), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu)), dEdu)
        return rev, diss

    def forward(self, u):
        """u: (B, 1, N) -> u_next: (B, 1, N). Dynamics from E, S, L, M.
        degeneracy_construction=True: exact dE/dt=0, dS/dt>=0 by construction."""
        B, C, N = u.shape
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        if self.l2_vargrad:
            scale = N / (2.0 * math.pi)
            dEdu = dEdu * scale
            dSdu = dSdu * scale

        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu)
            return u + rev + diss

        # --- legacy projection + correction path (ablation only) ---
        L_k, M_k = self._get_operators()
        m = L_k.shape[0]
        dEdu_hat = torch.fft.rfft(dEdu, dim=-1)
        dSdu_hat = torch.fft.rfft(dSdu, dim=-1)
        rev_hat = torch.zeros_like(dEdu_hat)
        rev_hat[:, :, :m] = L_k.unsqueeze(0).unsqueeze(0) * dEdu_hat[:, :, :m]
        rev = torch.fft.irfft(rev_hat, n=N)
        diss_hat = torch.zeros_like(dSdu_hat)
        diss_hat[:, :, :m] = M_k.unsqueeze(0).unsqueeze(0) * dSdu_hat[:, :, :m]
        diss = torch.fft.irfft(diss_hat, n=N)
        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual
        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        """Project du/dt ⊥ δE/δu → ensures dE/dt = ⟨δE/δu, du/dt⟩ = 0.
        Gram-Schmidt orthogonalization in function space."""
        # Inner products over spatial dimension
        inner_dudt_dE = (dudt * dEdu).sum(dim=-1, keepdim=True)  # (B, 1, 1)
        norm_dE_sq = (dEdu * dEdu).sum(dim=-1, keepdim=True) + 1e-10  # (B, 1, 1)

        # Remove component along δE/δu
        dudt_proj = dudt - (inner_dudt_dE / norm_dE_sq) * dEdu
        return dudt_proj

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        """Ensure dS/dt = ⟨δS/δu, du/dt⟩ ≥ 0.
        If violated, add a correction along δS/δu (projected ⊥ δE/δu)."""
        dSdt = (dSdu * dudt).sum(dim=-1, keepdim=True)  # (B, 1, 1)

        # Only correct when dS/dt < 0
        violation = F.relu(-dSdt)  # positive when dS/dt < 0

        if violation.sum() > 0:
            # Correction direction: δS/δu projected ⊥ δE/δu
            # (so correction doesn't break energy conservation)
            inner_SE = (dSdu * dEdu).sum(dim=-1, keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=-1, keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu

            norm_Sperp_sq = (dSdu_perp * dSdu_perp).sum(dim=-1, keepdim=True) + 1e-10

            # Add just enough correction to make dS/dt = 0 (from negative)
            # Need: ⟨δS/δu, dudt + α·dSdu_perp⟩ = 0
            # dSdt + α·⟨δS/δu, dSdu_perp⟩ = 0
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=-1, keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp  # only where violated

            dudt = dudt + alpha * dSdu_perp

        return dudt

    def predict_with_info(self, u):
        """Return prediction + thermodynamic info."""
        B, C, N = u.shape

        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        u_next = self.forward(u)

        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf)
        S_next = self.S_net(u_next_leaf)

        info = {
            'E': E.detach(), 'S': S.detach(),
            'E_next': E_next.detach(), 'S_next': S_next.detach(),
            'dEdu': dEdu.detach(), 'dSdu': dSdu.detach(),
            'dE': (E_next - E).detach(),
            'dS': (S_next - S).detach(),
        }
        return u_next, info

    def entropy_production(self, u):
        """Normalized entropy production r_S = <dS,du/dt>/(||dS|| ||du/dt||),
        differentiable. MEP penalty: prefer least-dissipative consistent fit."""
        N = u.shape[-1]
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        if self.l2_vargrad:
            scale = N / (2.0 * math.pi); dEdu = dEdu * scale; dSdu = dSdu * scale
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu)
            dudt = rev + diss
        else:
            L_k, M_k = self._get_operators(); m = L_k.shape[0]
            dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
            rh = torch.zeros_like(dEh); dh = torch.zeros_like(dSh)
            rh[:, :, :m] = L_k * dEh[:, :, :m]; dh[:, :, :m] = M_k * dSh[:, :, :m]
            dudt = torch.fft.irfft(rh, n=N) + torch.fft.irfft(dh, n=N)
            dudt = self._project_energy_conservation(dudt, dEdu)
            dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    def degeneracy_loss(self, u):
        """Soft regularizer L·δS/δu ≈ 0 and M·δE/δu ≈ 0 (legacy path only).
        Exact by construction when degeneracy_construction=True, so returns 0."""
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        L_k, M_k = self._get_operators()
        m = L_k.shape[0]

        dEdu_hat = torch.fft.rfft(dEdu, dim=-1)
        dSdu_hat = torch.fft.rfft(dSdu, dim=-1)

        # L · δS/δu should be ≈ 0
        L_dS = L_k.unsqueeze(0).unsqueeze(0) * dSdu_hat[:, :, :m]
        loss_L_deg = (L_dS.abs()**2).mean()

        # M · δE/δu should be ≈ 0
        M_dE = M_k.unsqueeze(0).unsqueeze(0) * dEdu_hat[:, :, :m]
        loss_M_deg = (M_dE.abs()**2).mean()

        return loss_L_deg + loss_M_deg


# ============================================================================
# Evaluation
# ============================================================================

def evaluate_model(model, data_in, data_out, pde_type, model_type='fno',
                   device='cpu', n_rollout=10):
    """Evaluate a trained model on all metrics.
    Note: no @torch.no_grad() because GENERIC-FNO needs autograd internally."""
    model.eval()
    model = model.to(device)
    data_in = data_in.to(device)
    data_out = data_out.to(device)

    is_time_dep = pde_type in ('heat', 'wave', 'burgers', 'advection')
    n_test = min(50, data_in.shape[0])

    results = {}

    # --- Single-step L2 error ---
    if is_time_dep:
        x = data_in[:n_test, 0:1, :]
        y = data_out[:n_test, 0:1, :]
    else:
        x = data_in[:n_test, 0:1, :]
        y = data_out[:n_test, 0:1, :]

    with torch.enable_grad():
        pred = model(x)
    l2_err = ((pred - y)**2).mean(dim=-1).sqrt() / ((y**2).mean(dim=-1).sqrt() + 1e-8)
    results['l2_single'] = l2_err.mean().item()

    if not is_time_dep:
        results['l2_rollout'] = results['l2_single']
        results['energy_track'] = 0.0
        results['mono_violations'] = 0.0
        results['dE_mean'] = 0.0
        results['dS_mean'] = 0.0
        return results

    # --- Multi-step rollout ---
    nt = min(n_rollout, data_in.shape[1])
    rollout_errors = []
    energy_traj_pred = []
    energy_traj_true = []

    x_roll = data_in[:n_test, 0:1, :].clone()
    for t in range(nt):
        with torch.enable_grad():
            x_roll = model(x_roll).detach()  # detach to avoid graph buildup
        y_t = data_out[:n_test, t:t+1, :]
        err = ((x_roll - y_t)**2).mean(dim=-1).sqrt() / ((y_t**2).mean(dim=-1).sqrt() + 1e-8)
        rollout_errors.append(err.mean().item())

        # Track energy
        E_pred = 0.5 * (x_roll**2).mean(dim=(-1, -2))  # (n_test,)
        E_true = 0.5 * (y_t**2).mean(dim=(-1, -2))
        energy_traj_pred.append(E_pred)
        energy_traj_true.append(E_true)

    results['l2_rollout'] = np.mean(rollout_errors)

    # --- Energy tracking ---
    E_pred_stack = torch.stack(energy_traj_pred, dim=1)  # (n_test, nt)
    E_true_stack = torch.stack(energy_traj_true, dim=1)
    energy_track_err = ((E_pred_stack - E_true_stack)**2).mean().sqrt().item()
    results['energy_track'] = energy_track_err

    # --- Monotonicity violations (for dissipative PDEs) ---
    if pde_type in ('heat', 'burgers'):
        E_init = 0.5 * (data_in[:n_test, 0:1, :]**2).mean(dim=(-1, -2))
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)  # (n_test, nt+1)
        dE = E_all[:, 1:] - E_all[:, :-1]  # should be ≤ 0 for dissipative
        violations = (dE > 1e-6).float().mean().item() * 100  # percentage
        results['mono_violations'] = violations
    elif pde_type in ('wave', 'advection'):
        # Energy should be roughly conserved
        E_init = 0.5 * (data_in[:n_test, 0:1, :]**2).mean(dim=(-1, -2))
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        E_variation = E_all.std(dim=1).mean().item()
        results['mono_violations'] = E_variation  # lower = better conservation
    else:
        results['mono_violations'] = 0.0

    # --- GENERIC-specific metrics ---
    if model_type == 'generic':
        with torch.enable_grad():
            x_check = data_in[:n_test, 0:1, :].clone()
            _, info = model.predict_with_info(x_check)
            results['dE_mean'] = info['dE'].mean().item()   # finite change (incl. curvature)
            results['dS_mean'] = info['dS'].mean().item()
            # First-order energy conservation rate (what the projection guarantees):
            #   r_E = |<dE/du, du/dt>| / (||dE/du|| ||du/dt||)   ~ machine zero
            u_leaf = x_check.detach().requires_grad_(True)
            E = model.E_net(u_leaf)
            dEdu = torch.autograd.grad(E.sum(), u_leaf)[0].detach()
            g = (model(x_check) - x_check).detach()
            num = (dEdu * g).sum(dim=(-1, -2)).abs()
            den = dEdu.flatten(1).norm(dim=1) * g.flatten(1).norm(dim=1) + 1e-12
            results['rE_mean'] = (num / den).mean().item()
    else:
        results['dE_mean'] = 0.0
        results['dS_mean'] = 0.0
        results['rE_mean'] = 0.0

    return results


# ============================================================================
# Main Benchmark
# ============================================================================

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



# ============================================================================
# Training (1D) — default e_sup_mode='none' to match the main paper
# ============================================================================

def train_model(model, data_in, data_out, pde_type, model_type='fno',
                n_epochs=150, lr=1e-3, batch_size=32, device='cpu',
                e_sup_mode='none', min_diss_weight=0.0):
    """1D trainer matching the 2D method: NO functional supervision by default.
    e_sup_mode in {'none','half_u2'} (half_u2 kept only for legacy/ablation)."""
    model = model.to(device)
    data_in = data_in.to(device)
    data_out = data_out.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

    n_samples = data_in.shape[0]
    is_time_dep = pde_type in ('heat', 'wave', 'burgers', 'advection')

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_samples, device=device)
        epoch_losses = defaultdict(float)
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i+batch_size]
            nt = data_in.shape[1]
            t_idx = torch.randint(0, nt, (1,)).item()
            x = data_in[idx, t_idx:t_idx+1, :].clone()
            y = data_out[idx, t_idx:t_idx+1, :].clone()

            if epoch > 30 and torch.rand(1).item() < 0.3:
                t_start = torch.randint(0, max(1, nt-2), (1,)).item()
                x = data_in[idx, t_start:t_start+1, :].clone()
                rollout_len = min(3, nt - t_start)
                pred = x
                rollout_loss = 0
                for step in range(rollout_len):
                    pred = model(pred)
                    target = data_out[idx, t_start+step:t_start+step+1, :]
                    rollout_loss += F.mse_loss(pred, target)
                loss = rollout_loss / rollout_len
                epoch_losses['rollout'] += loss.item()
                if model_type == 'generic':
                    deg = model.degeneracy_loss(x)
                    loss += min(1.0, epoch / 50.0) * 0.01 * deg
                    epoch_losses['degeneracy'] += deg.item()
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); n_batches += 1
                continue

            pred = model(x)
            loss = F.mse_loss(pred, y)
            epoch_losses['data'] += loss.item()

            if model_type == 'ep-fno':
                _, info = model.predict_with_info(x)
                ep = model.energy_penalty(info, pde_type)
                loss += min(1.0, epoch / 30.0) * 0.1 * ep
                epoch_losses['energy_penalty'] += ep.item()
            elif model_type == 'generic':
                deg = model.degeneracy_loss(x)
                loss += min(1.0, epoch / 50.0) * 0.01 * deg
                epoch_losses['degeneracy'] += deg.item()
                if min_diss_weight > 0.0:
                    mep = model.entropy_production(x)
                    loss += min(1.0, epoch / 50.0) * min_diss_weight * mep
                    epoch_losses['min_diss'] += mep.item()
                if e_sup_mode == 'half_u2':
                    u_leaf = x.detach().requires_grad_(True)
                    E_pred = model.E_net(u_leaf)
                    E_true = 0.5 * (x**2).mean(dim=(-1, -2))
                    e_sup = F.mse_loss(E_pred, E_true)
                    loss += max(0, 1.0 - epoch / 80.0) * 0.1 * e_sup
                    epoch_losses['E_supervision'] += e_sup.item()
                # 'none': no functional supervision

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); n_batches += 1

        scheduler.step()
        if (epoch + 1) % 50 == 0 or epoch == 0:
            avg = {k: v / max(n_batches, 1) for k, v in epoch_losses.items()}
            extras = ' '.join(f"{k}={v:.6f}" for k, v in avg.items())
            print(f"  Epoch {epoch+1:3d}: {extras}")

    return model


# ============================================================================
# 1D Benchmark (heat / wave / Burgers) — appendix
# ============================================================================

def channel_diagnostics(model, X, n_batch=40, Y=None):
    """Scale-invariant test of how much a trained GENERIC_FNO (1D) routes through
    the dissipative (M) channel. For the reversible wave equation a
    thermodynamically consistent model should give rho_M ~ 0 and r_S ~ 0.

      rho_M = ||diss|| / (||rev|| + ||diss||)
              fraction of the raw update du/dt = rev + diss from the M channel,
              rev = F^-1[L . F(dE/du)], diss = F^-1[M . F(dS/du)].
              Scale-invariant: rescaling S rescales M inversely, leaving diss fixed.
      r_S   = <dS/du, du/dt> / (||dS/du|| ||du/dt||)   normalized entropy production.
      r_E   = |<dE/du, du/dt>| / (||dE/du|| ||du/dt||)  projection check (~1e-6).
      L_mag, M_mag : RMS magnitude of the learned multipliers.
    """
    device = next(model.parameters()).device
    model.eval()
    X = X[:n_batch].to(device).detach()
    N = X.shape[-1]

    u_leaf = X.clone().requires_grad_(True)
    E = model.E_net(u_leaf); S = model.S_net(u_leaf)
    dEdu = torch.autograd.grad(E.sum(), u_leaf, retain_graph=True)[0].detach()
    dSdu = torch.autograd.grad(S.sum(), u_leaf)[0].detach()

    L_k, M_k = model._get_operators()
    L_k = L_k.detach(); M_k = M_k.detach(); m = L_k.shape[0]
    if getattr(model, 'degeneracy_construction', False):
        rev, diss = model._generic_rhs(dEdu, dSdu)
        rev = rev.detach(); diss = diss.detach()
    else:
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        rev_hat = torch.zeros_like(dEh); diss_hat = torch.zeros_like(dSh)
        rev_hat[:, :, :m] = L_k * dEh[:, :, :m]
        diss_hat[:, :, :m] = M_k * dSh[:, :, :m]
        rev = torch.fft.irfft(rev_hat, n=N); diss = torch.fft.irfft(diss_hat, n=N)

    L_mag = (model.a.detach() ** 2).mean().sqrt().item()
    M_mag = ((model.b_real.detach() ** 2 + model.b_imag.detach() ** 2) ** 2).mean().sqrt().item()

    def pnorm(z): return z.flatten(1).norm(dim=1)
    def ip(a, b): return (a * b).flatten(1).sum(dim=1)
    nrev, ndiss = pnorm(rev), pnorm(diss)
    rho_M = (ndiss / (nrev + ndiss + 1e-12)).mean().item()

    dudt = (model(X) - X).detach()
    r_S = (ip(dSdu, dudt) / (pnorm(dSdu) * pnorm(dudt) + 1e-12)).mean().item()
    r_E = (ip(dEdu, dudt).abs() / (pnorm(dEdu) * pnorm(dudt) + 1e-12)).mean().item()
    # gauge-invariant dissipation of fixed Q = 0.5||u||^2 (no learned E,S)
    r_mech = (-ip(X, dudt) / (pnorm(X) * pnorm(dudt) + 1e-12)).mean().item()
    Qx = (X ** 2).flatten(1).sum(dim=1)
    pi_model = ((Qx - (model(X).detach() ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    out = {'rho_M': rho_M, 'r_S': r_S, 'r_E': r_E, 'L_mag': L_mag, 'M_mag': M_mag,
           'r_mech': r_mech, 'pi_model': pi_model}
    if Y is not None:
        Yb = Y[:n_batch].to(device).detach()
        out['pi_true'] = ((Qx - (Yb ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    return out


def run_benchmark(save_dir=None, nx=64, n_samples=200, nt=20, n_epochs=150,
                  min_diss_weight=0.0, degeneracy_construction=True, seed=None,
                  pdes=('heat', 'advection', 'burgers', 'wave')):
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("=" * 70)
    print(f"GENERIC-FNO 1D BENCHMARK (no E-supervision)  device={device}")
    print(f"  degeneracy_construction={degeneracy_construction} "
          f"min_diss_weight={min_diss_weight}")
    print("=" * 70)
    WIDTH, MODES, N_LAYERS = 32, 16, 4
    WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC = 24, 12, 3
    BATCH = 32
    GENS = {'heat': generate_heat_data, 'wave': generate_wave_data,
            'advection': generate_advection_data,
            'burgers': generate_burgers_data}
    GENS = {k: GENS[k] for k in pdes}

    all_results = {}
    for pde, gen in GENS.items():
        print(f"\n{'='*50}\nPDE: {pde.upper()}\n{'='*50}")
        di, do, _ = gen(n_samples=n_samples, nx=nx, nt=nt)
        ntr = int(0.8 * n_samples)
        tr_in, te_in, tr_out, te_out = di[:ntr], di[ntr:], do[:ntr], do[ntr:]
        pde_res = {}
        for mname, mtype, ctor in [
            ('FNO', 'fno', lambda: VanillaFNO(WIDTH, MODES, N_LAYERS)),
            ('EP-FNO', 'ep-fno', lambda: EP_FNO(WIDTH, MODES, N_LAYERS)),
            ('GENERIC-FNO', 'generic', lambda: GENERIC_FNO(
                nx=nx, width_func=WIDTH_FUNC, modes_func=MODES_FUNC,
                n_layers_func=N_LAYERS_FUNC, modes_op=MODES,
                degeneracy_construction=degeneracy_construction)),
        ]:
            print(f"\n--- {mname} ---")
            model = ctor()
            t0 = time.time()
            model = train_model(model, tr_in, tr_out, pde, mtype, n_epochs=n_epochs,
                                lr=1e-3, batch_size=BATCH, device=device,
                                e_sup_mode='none',
                                min_diss_weight=(min_diss_weight if mtype == 'generic' else 0.0))
            r = evaluate_model(model, te_in, te_out, pde, mtype, device=device, n_rollout=10)
            r['time'] = time.time() - t0; r['params'] = count_params(model)
            pde_res[mname] = r
            print(f"  L2-1step={r['l2_single']:.6f} rollout={r['l2_rollout']:.6f} "
                  f"E-track={r['energy_track']:.6f} Mono={r['mono_violations']:.2f}")
            if mtype == 'generic':
                print(f"  r_E (1st-order)={r['rE_mean']:.2e}  "
                      f"dE/step={r['dE_mean']:.2e} dS/step={r['dS_mean']:.2e}")
                if save_dir:
                    import os
                    os.makedirs(save_dir, exist_ok=True)
                    ckpt = os.path.join(save_dir, f'generic_fno_1d_{pde}_nx{nx}'
                                        f'{"" if degeneracy_construction else "_legacy"}.pt')
                    torch.save({'state_dict': model.state_dict(), 'nx': nx,
                                'pde': pde, 'config': (WIDTH_FUNC, MODES_FUNC,
                                N_LAYERS_FUNC, MODES)}, ckpt)
                    print(f"  saved checkpoint -> {ckpt}")
                diag = channel_diagnostics(model, te_in[:, :1, :], n_batch=40,
                                           Y=te_out[:, :1, :])
                r['diagnostic'] = diag
                print(f"  [diag] rho_M={diag['rho_M']:.4f}  r_S={diag['r_S']:.2e}  "
                      f"r_E={diag['r_E']:.2e}  L_mag={diag['L_mag']:.2e}  "
                      f"M_mag={diag['M_mag']:.2e}")
                print(f"  [gauge-inv] r_mech={diag['r_mech']:+.4f}  "
                      f"pi_model={diag['pi_model']:+.4e}  "
                      f"pi_true={diag.get('pi_true', float('nan')):+.4e}")
        all_results[pde] = pde_res

    # summary + LaTeX rows
    print("\n" + "=" * 70 + "\n1D RESULTS (no E-supervision)\n" + "=" * 70)
    for pde, pr in all_results.items():
        print(f"\n{pde.upper()}")
        print(f"  {'Model':<13}{'Params':>9}{'L2-1step':>11}{'L2-roll':>11}{'E-track':>11}{'Mono%':>8}")
        for mn, r in pr.items():
            print(f"  {mn:<13}{r['params']:>9,}{r['l2_single']:>11.6f}"
                  f"{r['l2_rollout']:>11.6f}{r['energy_track']:>11.6f}{r['mono_violations']:>8.2f}")
        if 'GENERIC-FNO' in pr:
            f_, g_ = pr['FNO'], pr['GENERIC-FNO']
            d = (g_['l2_rollout']-f_['l2_rollout'])/(f_['l2_rollout']+1e-10)*100
            print(f"  -> GENERIC vs FNO L2-roll: {d:+.1f}%  "
                  f"r_E={g_['rE_mean']:.1e}  dE/step={g_['dE_mean']:.1e} dS/step={g_['dS_mean']:.1e}")

    print("\n=== LaTeX rows (PDE / Model / Params / L2-1step / L2-roll / E-track / Mono) ===\n")
    for pde, pr in all_results.items():
        print(f"\\multirow{{3}}{{*}}{{{pde.capitalize()}}}")
        for mn in ['FNO', 'EP-FNO', 'GENERIC-FNO']:
            if mn not in pr: continue
            r = pr[mn]
            params = f"{r['params']/1e3:.0f}K"
            print(f" & {mn:<11} & {params:>5} & {r['l2_single']:.4f} & "
                  f"{r['l2_rollout']:.4f} & {r['energy_track']:.4f} & "
                  f"{r['mono_violations']:.2f} \\\\")
        print("\\midrule")

    # --- generator-channel diagnostic summary ---
    print("\n" + "=" * 70)
    print("GENERATOR-CHANNEL DIAGNOSTIC (scale-invariant; wave should be ~0)")
    print("=" * 70)
    print(f"  {'PDE':<9}{'rho_M':>10}{'r_S':>12}{'r_E':>12}{'L_mag':>11}{'M_mag':>11}")
    print("  " + "-" * 64)
    for pde, pr in all_results.items():
        if 'GENERIC-FNO' in pr and 'diagnostic' in pr['GENERIC-FNO']:
            d = pr['GENERIC-FNO']['diagnostic']
            print(f"  {pde:<9}{d['rho_M']:>10.4f}{d['r_S']:>12.2e}"
                  f"{d['r_E']:>12.2e}{d['L_mag']:>11.3e}{d['M_mag']:>11.3e}")
    print("\n  LaTeX rows (PDE / rho_M / r_S):")
    for pde, pr in all_results.items():
        if 'GENERIC-FNO' in pr and 'diagnostic' in pr['GENERIC-FNO']:
            d = pr['GENERIC-FNO']['diagnostic']
            print(f"  {pde.capitalize():<8} & {d['rho_M']:.3f} & {d['r_S']:.1e} \\\\")

    if save_dir:
        import os
        os.makedirs(save_dir, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'benchmark_1d_none_{ts}.pkl'), 'wb') as f:
            pickle.dump({'results': all_results, 'nx': nx, 'device': device}, f)
        print(f"\nSaved: {save_dir}/benchmark_1d_none_{ts}.pkl + GENERIC checkpoints")
    return all_results


def run_benchmark_seeds(n_seeds=3, save_dir=None, **kwargs):
    """Run run_benchmark over n_seeds (data + initialization resampled each seed)
    and report mean +/- std. kwargs forwarded to run_benchmark. Checkpoints on
    seed 0. Returns the list of per-seed all_results dicts."""
    import numpy as _np
    runs = []
    for s in range(n_seeds):
        print("\n" + "#" * 80 + f"\n# SEED {s}/{n_seeds-1}\n" + "#" * 80)
        runs.append(run_benchmark(seed=s, save_dir=(save_dir if s == 0 else None), **kwargs))

    pdes = list(runs[0].keys())
    models = list(runs[0][pdes[0]].keys())

    def ms(pde, model, key, sub=None):
        vals = []
        for r in runs:
            d = r[pde][model]
            if sub:
                d = d.get(sub, {})
            v = d.get(key)
            if v is not None:
                vals.append(v)
        a = _np.array(vals, float)
        return (float(a.mean()), float(a.std())) if a.size else (float('nan'), float('nan'))

    L = ["", "=" * 84, f"SEED-AVERAGED RESULTS  (n_seeds={n_seeds}, mean +/- std)", "=" * 84]
    for pde in pdes:
        L.append(f"\n{pde.upper()}")
        L.append(f"  {'Model':<14}{'params':>9}{'L2-1step':>18}{'L2-roll':>18}")
        for m in models:
            p = runs[0][pde][m].get('params', 0)
            sm, ss = ms(pde, m, 'l2_single'); rm, rs = ms(pde, m, 'l2_rollout')
            L.append(f"  {m:<14}{p:>9,}{sm:>9.4f}+/-{ss:<6.4f}{rm:>9.4f}+/-{rs:<6.4f}")

    gname = next((m for m in models if 'GENERIC' in m), None)
    if gname:
        L += ["", "GAUGE-INVARIANT DISSIPATION  (GENERIC, mean +/- std)",
              f"  {'PDE':<10}{'r_mech':>16}{'pi_model':>20}{'pi_true':>20}"]
        for pde in pdes:
            rm, rs = ms(pde, gname, 'r_mech', 'diagnostic')
            pm, ps = ms(pde, gname, 'pi_model', 'diagnostic')
            tm, ts = ms(pde, gname, 'pi_true', 'diagnostic')
            L.append(f"  {pde:<10}{rm:>+8.3f}+/-{rs:<6.3f}{pm:>+10.2e}+/-{ps:<8.2e}{tm:>+10.2e}")
        L += ["", "GAUGE-DEPENDENT MECHANISM  (GENERIC, mean +/- std)",
              f"  {'PDE':<10}{'rho_M':>16}{'r_S':>16}{'r_E':>14}"]
        for pde in pdes:
            rm, rs = ms(pde, gname, 'rho_M', 'diagnostic')
            sm, ss = ms(pde, gname, 'r_S', 'diagnostic')
            em, es = ms(pde, gname, 'r_E', 'diagnostic')
            L.append(f"  {pde:<10}{rm:>8.3f}+/-{rs:<6.3f}{sm:>8.3f}+/-{ss:<6.3f}{em:>12.1e}")

    L += ["", "  LaTeX accuracy rows (PDE & Model & params & L2-1step & L2-roll):"]
    for pde in pdes:
        for m in models:
            p = runs[0][pde][m].get('params', 0)
            sm, ss = ms(pde, m, 'l2_single'); rm, rs = ms(pde, m, 'l2_rollout')
            L.append(f"  {pde.capitalize()} & {m} & {p/1e3:.0f}K & "
                     f"${sm:.3f}\\pm{ss:.3f}$ & ${rm:.3f}\\pm{rs:.3f}$ \\\\")
    if gname:
        L += ["", "  LaTeX gauge-invariant rows (PDE & r_mech & pi_model & pi_true):"]
        for pde in pdes:
            rm, rs = ms(pde, gname, 'r_mech', 'diagnostic')
            pm, ps = ms(pde, gname, 'pi_model', 'diagnostic')
            tm, _ = ms(pde, gname, 'pi_true', 'diagnostic')
            L.append(f"  {pde.capitalize()} & ${rm:+.3f}\\pm{rs:.3f}$ & "
                     f"${pm:+.2e}$ & ${tm:+.2e}$ \\\\")

    summary = "\n".join(L)
    print(summary)
    if save_dir:
        import os, pickle, time
        os.makedirs(save_dir, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'benchmark_1d_seeds_{ts}.pkl'), 'wb') as f:
            pickle.dump({'runs': runs, 'n_seeds': n_seeds}, f)
        with open(os.path.join(save_dir, f'benchmark_1d_seeds_{ts}.txt'), 'w') as f:
            f.write(summary)
        print(f"\nSaved seed-averaged summary + per-seed results -> {save_dir}")
    return runs


def run_in_colab():
    import os
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        save_dir = '/content/drive/MyDrive/GENERIC_FNO_results'
    except ImportError:
        save_dir = './GENERIC_FNO_results'
    os.makedirs(save_dir, exist_ok=True)
    print(f"Results -> {save_dir}\n")
    return run_benchmark(save_dir=save_dir)


if __name__ == '__main__':
    #run_in_colab()
    DRIVE = '/content/drive/MyDrive/GENERIC_FNO_results'

    # 1D FNO (~fast)
    run_benchmark_seeds(n_seeds=3, save_dir=DRIVE)


################################################################################
# SEED 0/2
################################################################################
GENERIC-FNO 1D BENCHMARK (no E-supervision)  device=cuda
  degeneracy_construction=True min_diss_weight=0.0

PDE: HEAT

--- FNO ---
  Epoch   1: data=0.017076
  Epoch  50: data=0.000071 rollout=0.000105
  Epoch 100: data=0.000004 rollout=0.000011
  Epoch 150: data=0.000006
  L2-1step=0.019207 rollout=0.095538 E-track=0.017949 Mono=7.50

--- EP-FNO ---
  Epoch   1: data=0.003233 energy_penalty=0.000176
  Epoch  50: data=0.000035 energy_penalty=0.000000 rollout=0.000072
  Epoch 100: data=0.000002 energy_penalty=0.000000 rollout=0.000009
  Epoch 150: rollout=0.000008 data=0.000002 energy_penalty=0.000000
  L2-1step=0.015549 rollout=0.074343 E-track=0.015053 Mono=8.00

--- GENERIC-FNO ---
  Epoch   1: data=0.000041 degeneracy=0.000000
  Epoch  50: rollout=0.000048 degeneracy=0.000000 data=0.000004
  Epoch 100: data=0.0

In [ ]:
#!/usr/bin/env python3
"""
GENERIC-FNO Benchmark
=====================
Fourier Neural Operator with GENERIC thermodynamic structure.

Architecture:
  - E-net (FNO → scalar): learns energy functional E[u]
  - S-net (FNO → scalar): learns entropy functional S[u]
  - L(k): anti-Hermitian diagonal operator (reversible dynamics)
  - M(k): Hermitian PSD diagonal operator (dissipative dynamics)
  - Dynamics: du/dt = L·δE/δu + M·δS/δu
  - Hard projection: energy conservation + entropy non-decrease

Compares: FNO (vanilla), EP-FNO (energy penalty), GENERIC-FNO (ours)
Tests on: Heat, Wave, Burgers, Poisson, Biharmonic

Key insight: dynamics are CONSTRUCTED from E,S — no bypass possible.
Unlike prior FMO approach where physics was a soft auxiliary loss on an
unconstrained backbone, here the model MUST learn meaningful E and S
to produce correct predictions.

Reference gap: No prior work embeds full GENERIC (energy conservation +
entropy production) into function-space neural operators (FNO/DeepONet).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import time
import math
from collections import defaultdict

# ============================================================================
# Data Generation (same 5 PDEs)
# ============================================================================

def generate_heat_data(n_samples=200, nx=64, nt=20, dt=0.01, nu=0.01):
    """Heat equation: du/dt = nu * d²u/dx². Purely dissipative."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    decay = torch.exp(-nu * k**2 * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)  # (nt+1, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'heat'

def generate_wave_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0):
    """Wave equation (1st order system): du/dt = c*dv/dx, dv/dt = c*du/dx.
    Purely reversible (Hamiltonian). We track u-component only."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    omega = c * k  # dispersion relation

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        v0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)
            # Give some initial velocity too
            amp_v = torch.randn(1).item() * 0.3
            v0 += amp_v * torch.cos(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        v_hat = torch.fft.rfft(v0)

        traj = [u0.clone()]
        for t in range(nt):
            # Exact solution: rotate in (u_hat, v_hat) space
            cos_w = torch.cos(omega * dt)
            sin_w = torch.sin(omega * dt)
            u_new = cos_w * u_hat + 1j * sin_w * v_hat
            v_new = 1j * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'wave'

def generate_advection_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0, max_mode=6):
    """1D linear advection du/dt + c*du/dx = 0. Reversible, Markovian in u,
    conserves 0.5<u^2> exactly => thermodynamically-consistent model drives M->0."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    phase = torch.exp(-1j * c * k * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, max_mode+1, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            ph = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + ph)
        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft(u_hat, n=nx))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'

def generate_burgers_data(n_samples=200, nx=64, nt=20, dt=0.005, nu=0.02):
    """Viscous Burgers: du/dt + u*du/dx = nu*d²u/dx². Mixed rev+diss."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    dx = x[1] - x[0]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, 5, (1,)).item()
            amp = torch.randn(1).item() * 0.3
            phase = torch.rand(1).item() * 2 * math.pi
            u += amp * torch.sin(ki * x + phase)

        traj = [u.clone()]
        # Semi-implicit: diffusion in spectral, advection in physical
        for t in range(nt):
            u_hat = torch.fft.rfft(u)
            # Diffusion (implicit)
            u_hat = u_hat / (1 + nu * k**2 * dt)
            u = torch.fft.irfft(u_hat, n=nx)
            # Advection (explicit, spectral derivative)
            du_dx = torch.fft.irfft(1j * k * torch.fft.rfft(u), n=nx)
            u = u - dt * u * du_dx
            traj.append(u.clone())

        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'burgers'



# ============================================================================
# DeepONet building blocks
# ============================================================================

def _mlp(sizes, act=nn.GELU):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2:
            layers.append(act())
    return nn.Sequential(*layers)


class DeepONetField(nn.Module):
    """DeepONet mapping a field u (B,1,N) to a field G(u) (B,1,N) on the same grid.
       branch: u sampled at the N grid sensors -> p coefficients;
       trunk:  grid coordinate -> p basis features; G(u)(y)=sum_k b_k(u) tau_k(y)."""
    def __init__(self, nx, p=64, hidden=128):
        super().__init__()
        self.nx, self.p = nx, p
        self.branch = _mlp([nx, hidden, hidden, p])
        self.trunk = _mlp([1, hidden, hidden, p])
        self.bias = nn.Parameter(torch.zeros(1))
        coords = torch.linspace(0, 1, nx + 1)[:-1].unsqueeze(1)  # (N,1)
        self.register_buffer('coords', coords)

    def forward(self, u):                      # u: (B,1,N)
        b = self.branch(u.squeeze(1))          # (B,p)
        t = self.trunk(self.coords)            # (N,p)
        G = b @ t.transpose(0, 1) + self.bias  # (B,N)
        return G.unsqueeze(1)                  # (B,1,N)


class VanillaDeepONet1d(nn.Module):
    """Baseline: residual operator u_{t+1} = u_t + G(u_t) via DeepONet."""
    def __init__(self, nx, p=64, hidden=128):
        super().__init__()
        self.net = DeepONetField(nx, p, hidden)

    def forward(self, u):
        return u + self.net(u)

    def predict_with_info(self, u):
        return self.forward(u), {}

    def energy_penalty(self, info, pde_type):  # for API parity (unused)
        return torch.tensor(0.0, device=next(self.parameters()).device)


class DeepONetFunctional(nn.Module):
    """Scalar functional E[u] via DeepONet: field G(u)(y) -> spatial mean -> MLP head."""
    def __init__(self, nx, p=64, hidden=128):
        super().__init__()
        self.field = DeepONetField(nx, p, hidden)
        self.head = _mlp([1, 32, 1])

    def forward(self, u):                  # (B,1,N) -> (B,)
        G = self.field(u)                  # (B,1,N)
        m = G.mean(dim=(-1, -2))           # (B,)  integral functional
        return self.head(m.unsqueeze(-1)).squeeze(-1)


# ============================================================================
# GENERIC-DeepONet: DeepONet functionals + (reused) GENERIC dynamics in 1D
# ============================================================================

class GENERIC_DeepONet1d(nn.Module):
    def __init__(self, nx, p=64, hidden=128, modes_op=16,
                 degeneracy_construction=True):
        super().__init__()
        self.nx = nx
        self.degeneracy_construction = degeneracy_construction
        self.E_net = DeepONetFunctional(nx, p, hidden)
        self.S_net = DeepONetFunctional(nx, p, hidden)
        m = min(modes_op, nx // 2 + 1)
        self.m = m
        # L = i a (skew); M = |b|^2 (PSD); diagonal Fourier multipliers on low modes
        self.a = nn.Parameter(0.3 * torch.randn(m))
        self.b_r = nn.Parameter(0.3 * torch.randn(m))
        self.b_i = nn.Parameter(0.3 * torch.randn(m))
        self.residual = DeepONetField(nx, p, hidden)
        self.residual_gate = nn.Parameter(torch.tensor(-3.0))

    def _ops(self):
        return 1j * self.a, self.b_r**2 + self.b_i**2

    def _apply_ops(self, dEh, dSh):
        m = self.m
        L, M = self._ops()
        rev = torch.zeros_like(dEh)
        diss = torch.zeros_like(dSh)
        rev[:, :, :m] = L * dEh[:, :, :m]
        diss[:, :, :m] = M * dSh[:, :, :m]
        return rev, diss

    # --- degeneracy-by-construction helpers ---
    def _L_apply(self, v):
        m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        out[:, :, :m] = (1j * self.a) * vh[:, :, :m]
        return torch.fft.irfft(out, n=self.nx, dim=-1)

    def _M_apply(self, v):
        m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        out[:, :, :m] = (self.b_r**2 + self.b_i**2) * vh[:, :, :m]
        return torch.fft.irfft(out, n=self.nx, dim=-1)

    def _remove(self, v, w):
        return v - (self._ip(v, w) / (self._ip(w, w) + 1e-12)) * w

    def _generic_rhs(self, dEdu, dSdu):
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu)), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu)), dEdu)
        return rev, diss

    def entropy_production(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu); dudt = rev + diss
        else:
            dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
            rev, diss = self._apply_ops(dEh, dSh)
            dudt = torch.fft.irfft(rev, n=self.nx, dim=-1) + torch.fft.irfft(diss, n=self.nx, dim=-1)
            dudt = self._proj_E(dudt, dEdu); dudt = self._ensure_S(dudt, dSdu, dEdu)
        num = (dSdu * dudt).flatten(1).sum(dim=1)
        den = dSdu.flatten(1).norm(dim=1) * dudt.flatten(1).norm(dim=1) + 1e-8
        return (num / den).clamp(min=0).mean()

    @staticmethod
    def _ip(a, b):
        return (a * b).sum(dim=(-1, -2), keepdim=True)

    def _proj_E(self, g, dEdu):
        return g - (self._ip(g, dEdu) / (self._ip(dEdu, dEdu) + 1e-12)) * dEdu

    def _ensure_S(self, g, dSdu, dEdu):
        s_perp = dSdu - (self._ip(dSdu, dEdu) / (self._ip(dEdu, dEdu) + 1e-12)) * dEdu
        ip_sg = self._ip(dSdu, g)
        denom = self._ip(dSdu, s_perp) + 1e-12
        alpha = torch.relu(-ip_sg) / denom
        return g + alpha * s_perp

    def degeneracy_loss(self, u):
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        L, M = self._ops(); m = self.m
        L_dS = L * dSh[:, :, :m]
        M_dE = M * dEh[:, :, :m]
        return (L_dS.abs()**2).mean() + (M_dE.abs()**2).mean()

    def forward(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        if self.degeneracy_construction:
            # degeneracy by construction: dE/dt=0, dS/dt>=0 exactly, no residual
            rev, diss = self._generic_rhs(dEdu, dSdu)
            return u + rev + diss
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        rev, diss = self._apply_ops(dEh, dSh)
        rev = torch.fft.irfft(rev, n=self.nx, dim=-1)
        diss = torch.fft.irfft(diss, n=self.nx, dim=-1)
        dudt = rev + diss
        dudt = self._proj_E(dudt, dEdu)
        dudt = self._ensure_S(dudt, dSdu, dEdu)
        gate = torch.sigmoid(self.residual_gate)
        res = self._proj_E(gate * self.residual(u_leaf), dEdu)
        return u + dudt + res

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        un = self.forward(u)
        with torch.no_grad():
            dE = (self.E_net(un) - E)
            dS = (self.S_net(un) - S)
        return un, {'dE': dE, 'dS': dS}


# ============================================================================
# Train / evaluate
# ============================================================================

def train_model(model, di, do, pde, model_type='deeponet', n_epochs=150,
                lr=1e-3, batch_size=32, device='cpu', min_diss_weight=0.0):
    model = model.to(device); di = di.to(device); do = do.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_epochs)
    n = di.shape[0]; nt = di.shape[1]
    for ep in range(n_epochs):
        model.train(); perm = torch.randperm(n, device=device); losses = defaultdict(float); nb = 0
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            t = torch.randint(0, nt, (1,)).item()
            x = di[idx, t:t+1, :].clone(); y = do[idx, t:t+1, :].clone()
            if ep > 30 and torch.rand(1).item() < 0.3:
                ts = torch.randint(0, max(1, nt-2), (1,)).item()
                x = di[idx, ts:ts+1, :].clone(); rl = min(3, nt-ts); pred = x; rloss = 0
                for s in range(rl):
                    pred = model(pred); rloss += F.mse_loss(pred, do[idx, ts+s:ts+s+1, :])
                loss = rloss/rl; losses['rollout'] += loss.item()
                if model_type == 'generic':
                    deg = model.degeneracy_loss(x); loss += min(1.0, ep/50.)*0.01*deg
                    losses['degeneracy'] += deg.item()
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); nb += 1; continue
            pred = model(x); loss = F.mse_loss(pred, y); losses['data'] += loss.item()
            if model_type == 'generic':
                deg = model.degeneracy_loss(x); loss += min(1.0, ep/50.)*0.01*deg
                losses['degeneracy'] += deg.item()
                if min_diss_weight > 0.0:
                    mep = model.entropy_production(x)
                    loss += min(1.0, ep/50.) * min_diss_weight * mep
                    losses['min_diss'] += mep.item()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); nb += 1
        sched.step()
        if (ep+1) % 50 == 0 or ep == 0:
            print("  Epoch %3d: %s" % (ep+1, ' '.join(f"{k}={v/max(nb,1):.6f}" for k,v in losses.items())))
    return model


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def evaluate_model(model, di, do, pde, model_type='deeponet', device='cpu', n_rollout=10):
    model.eval(); di = di.to(device); do = do.to(device)
    nte = di.shape[0]
    x = di[:, 0:1, :]; y = do[:, 0:1, :]
    pred = model(x)
    l2_single = (((pred - y)**2).sum(dim=(-1,-2)).sqrt() /
                 ((y**2).sum(dim=(-1,-2)).sqrt() + 1e-12)).mean().item()
    # rollout
    u = di[:, 0:1, :]; errs = []; energies = []
    mono_viol = 0; mono_tot = 0
    for t in range(min(n_rollout, di.shape[1])):
        u = model(u); tgt = do[:, t:t+1, :]
        errs.append((((u-tgt)**2).sum(dim=(-1,-2)).sqrt() /
                     ((tgt**2).sum(dim=(-1,-2)).sqrt()+1e-12)).mean().item())
        e = 0.5*(u**2).mean(dim=(-1,-2))
        energies.append(e.detach())
        if pde in ('heat', 'burgers') and t > 0:
            mono_viol += (energies[-1] > energies[-2] + 1e-9).sum().item(); mono_tot += e.numel()
    l2_rollout = float(np.mean(errs))
    # energy tracking
    et = []
    for t in range(min(n_rollout, di.shape[1])):
        et.append((0.5*(do[:, t:t+1, :]**2).mean(dim=(-1,-2))).detach())
    E_pred = torch.stack(energies, 1); E_true = torch.stack(et, 1)
    energy_track = ((E_pred - E_true)**2).mean().sqrt().item()
    mono = 100.0*mono_viol/max(mono_tot, 1)
    res = {'l2_single': l2_single, 'l2_rollout': l2_rollout,
           'energy_track': energy_track, 'mono_violations': mono}
    if model_type == 'generic':
        with torch.enable_grad():
            xc = di[:, 0:1, :].clone()
            _, info = model.predict_with_info(xc)
            res['dE_mean'] = info['dE'].mean().item(); res['dS_mean'] = info['dS'].mean().item()
            u_leaf = xc.detach().requires_grad_(True)
            E = model.E_net(u_leaf)
            dEdu = torch.autograd.grad(E.sum(), u_leaf)[0].detach()
            g = (model(xc) - xc).detach()
            num = (dEdu*g).sum(dim=(-1,-2)).abs()
            den = dEdu.flatten(1).norm(dim=1)*g.flatten(1).norm(dim=1)+1e-12
            res['rE_mean'] = (num/den).mean().item()
    else:
        res['dE_mean'] = res['dS_mean'] = res['rE_mean'] = 0.0
    return res


# ============================================================================
# Benchmark: DeepONet vs GENERIC-DeepONet (1D)
# ============================================================================

def channel_diagnostics(model, X, n_batch=40, Y=None):
    """Scale-invariant L-vs-M usage for GENERIC-DeepONet1d (matches the FNO diag).
    rho_M = ||M dS|| / (||L dE|| + ||M dS||);  r_S = <dS,du/dt>/(||dS|| ||du/dt||);
    r_E = |<dE,du/dt>|/(||dE|| ||du/dt||).  Reversible (advection) => rho_M, r_S ~ 0."""
    device = next(model.parameters()).device
    model.eval(); X = X[:n_batch].to(device).detach(); N = X.shape[-1]
    u_leaf = X.clone().requires_grad_(True)
    E = model.E_net(u_leaf); S = model.S_net(u_leaf)
    dEdu = torch.autograd.grad(E.sum(), u_leaf, retain_graph=True)[0].detach()
    dSdu = torch.autograd.grad(S.sum(), u_leaf)[0].detach()
    if getattr(model, 'degeneracy_construction', False):
        rev, diss = model._generic_rhs(dEdu, dSdu); rev = rev.detach(); diss = diss.detach()
    else:
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        rh, dh = model._apply_ops(dEh, dSh)
        rev = torch.fft.irfft(rh, n=N, dim=-1); diss = torch.fft.irfft(dh, n=N, dim=-1)
    a2 = (model.a.detach()**2).mean().sqrt().item()
    M_mag = (((model.b_r.detach()**2 + model.b_i.detach()**2)**2).mean()).sqrt().item()
    def pn(z): return z.flatten(1).norm(dim=1)
    def ip(a, b): return (a*b).flatten(1).sum(dim=1)
    rho_M = (pn(diss)/(pn(rev)+pn(diss)+1e-12)).mean().item()
    dudt = (model(X)-X).detach()
    r_S = (ip(dSdu, dudt)/(pn(dSdu)*pn(dudt)+1e-12)).mean().item()
    r_E = (ip(dEdu, dudt).abs()/(pn(dEdu)*pn(dudt)+1e-12)).mean().item()
    # gauge-invariant dissipation of fixed Q = 0.5||u||^2 (no learned E,S)
    r_mech = (-ip(X, dudt)/(pn(X)*pn(dudt)+1e-12)).mean().item()
    Qx = (X**2).flatten(1).sum(dim=1)
    out = {'rho_M': rho_M, 'r_S': r_S, 'r_E': r_E, 'L_mag': a2, 'M_mag': M_mag,
           'r_mech': r_mech}
    if Y is not None:
        Yb = Y[:n_batch].to(device).detach()
        out['pi_true'] = ((Qx - (Yb**2).flatten(1).sum(dim=1))/(Qx+1e-12)).mean().item()
    return out


def run_benchmark(save_dir=None, nx=64, n_samples=200, nt=20, n_epochs=150,
                  min_diss_weight=0.0, degeneracy_construction=True, seed=None,
                  pdes=('heat', 'advection', 'burgers', 'wave')):
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("="*70); print(f"GENERIC-DeepONet 1D STUDY (no E-supervision)  device={device}"); print("="*70)
    P, HID, MODES = 64, 128, 16
    print(f"  degeneracy_construction={degeneracy_construction} min_diss_weight={min_diss_weight}")
    GENS = {'heat': generate_heat_data, 'wave': generate_wave_data,
            'advection': generate_advection_data, 'burgers': generate_burgers_data}
    GENS = {k: GENS[k] for k in pdes}
    allres = {}
    for pde, gen in GENS.items():
        print(f"\n{'='*50}\nPDE: {pde.upper()}\n{'='*50}")
        di, do, _ = gen(n_samples=n_samples, nx=nx, nt=nt)
        ntr = int(0.8*n_samples)
        tri, tei, tro, teo = di[:ntr], di[ntr:], do[:ntr], do[ntr:]
        pr = {}
        for name, mt, ctor in [
            ('DeepONet', 'deeponet', lambda: VanillaDeepONet1d(nx, P, HID)),
            ('GENERIC-DeepONet', 'generic', lambda: GENERIC_DeepONet1d(
                nx, P, HID, MODES, degeneracy_construction=degeneracy_construction)),
        ]:
            print(f"\n--- {name} ---")
            model = ctor(); t0 = time.time()
            model = train_model(model, tri, tro, pde, mt, n_epochs=n_epochs, batch_size=32,
                                device=device,
                                min_diss_weight=(min_diss_weight if mt == 'generic' else 0.0))
            r = evaluate_model(model, tei, teo, pde, mt, device=device, n_rollout=10)
            r['time'] = time.time()-t0; r['params'] = count_params(model); pr[name] = r
            print(f"  L2-1step={r['l2_single']:.6f} rollout={r['l2_rollout']:.6f} "
                  f"E-track={r['energy_track']:.6f} Mono={r['mono_violations']:.2f}")
            if mt == 'generic':
                print(f"  r_E={r['rE_mean']:.2e} dE/step={r['dE_mean']:.2e} dS/step={r['dS_mean']:.2e}")
                if save_dir:
                    import os; os.makedirs(save_dir, exist_ok=True)
                    tagd = '' if degeneracy_construction else '_legacy'
                    ckpt = os.path.join(save_dir, f'generic_deeponet_1d_{pde}_nx{nx}{tagd}.pt')
                    torch.save({'state_dict': model.state_dict(), 'nx': nx, 'pde': pde,
                                'degeneracy_construction': degeneracy_construction}, ckpt)
                    print(f"  saved checkpoint -> {ckpt}")
                diag = channel_diagnostics(model, tei[:, :1, :], n_batch=40, Y=teo[:, :1, :])
                r['diagnostic'] = diag
                print(f"  [diag] rho_M={diag['rho_M']:.4f} r_S={diag['r_S']:.2e} "
                      f"r_E={diag['r_E']:.2e} L_mag={diag['L_mag']:.2e} M_mag={diag['M_mag']:.2e}")
                print(f"  [gauge-inv] r_mech={diag['r_mech']:+.4f} "
                      f"pi_true={diag.get('pi_true', float('nan')):+.4e}")
        allres[pde] = pr

    print("\n"+"="*70+"\n1D DeepONet STUDY RESULTS\n"+"="*70)
    for pde, pr in allres.items():
        print(f"\n{pde.upper()}")
        for mn, r in pr.items():
            print(f"  {mn:<18}{r['params']:>9,}  L2roll={r['l2_rollout']:.4f}  Mono={r['mono_violations']:.2f}")
        if 'GENERIC-DeepONet' in pr and 'DeepONet' in pr:
            d = pr['DeepONet']; g = pr['GENERIC-DeepONet']
            imp = (g['l2_rollout']-d['l2_rollout'])/(d['l2_rollout']+1e-10)*100
            print(f"  -> GENERIC-DeepONet vs DeepONet L2-roll: {imp:+.1f}%  r_E={g['rE_mean']:.1e}")

    print("\n=== LaTeX rows (PDE / Model / Params / L2-1step / L2-roll / E-track / Mono) ===\n")
    for pde, pr in allres.items():
        print(f"\\multirow{{2}}{{*}}{{{pde.capitalize()}}}")
        for mn in ['DeepONet', 'GENERIC-DeepONet']:
            r = pr[mn]; params = f"{r['params']/1e3:.0f}K"
            print(f" & {mn:<17} & {params:>5} & {r['l2_single']:.4f} & {r['l2_rollout']:.4f} & "
                  f"{r['energy_track']:.4f} & {r['mono_violations']:.2f} \\\\")
        print("\\midrule")

    if save_dir:
        import os; os.makedirs(save_dir, exist_ok=True); ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'deeponet_1d_{ts}.pkl'), 'wb') as f:
            pickle.dump({'results': allres, 'nx': nx, 'device': device}, f)
        print(f"\nSaved: {save_dir}/deeponet_1d_{ts}.pkl")
    return allres


def run_benchmark_seeds(n_seeds=3, save_dir=None, **kwargs):
    """Run run_benchmark over n_seeds (data + initialization resampled each seed)
    and report mean +/- std. kwargs forwarded to run_benchmark. Checkpoints on
    seed 0. Returns the list of per-seed result dicts."""
    import numpy as _np
    runs = []
    for s in range(n_seeds):
        print("\n" + "#" * 80 + f"\n# SEED {s}/{n_seeds-1}\n" + "#" * 80)
        runs.append(run_benchmark(seed=s, save_dir=(save_dir if s == 0 else None), **kwargs))

    pdes = list(runs[0].keys())
    models = list(runs[0][pdes[0]].keys())

    def ms(pde, model, key, sub=None):
        vals = []
        for r in runs:
            d = r[pde][model]
            if sub:
                d = d.get(sub, {})
            v = d.get(key)
            if v is not None:
                vals.append(v)
        a = _np.array(vals, float)
        return (float(a.mean()), float(a.std())) if a.size else (float('nan'), float('nan'))

    L = ["", "=" * 84, f"SEED-AVERAGED RESULTS  (n_seeds={n_seeds}, mean +/- std)", "=" * 84]
    for pde in pdes:
        L.append(f"\n{pde.upper()}")
        L.append(f"  {'Model':<18}{'params':>9}{'L2-1step':>18}{'L2-roll':>18}")
        for m in models:
            p = runs[0][pde][m].get('params', 0)
            sm, ss = ms(pde, m, 'l2_single'); rm, rs = ms(pde, m, 'l2_rollout')
            L.append(f"  {m:<18}{p:>9,}{sm:>9.4f}+/-{ss:<6.4f}{rm:>9.4f}+/-{rs:<6.4f}")

    gname = next((m for m in models if 'GENERIC' in m), None)
    if gname:
        L += ["", "GAUGE-INVARIANT DISSIPATION  (GENERIC r_mech vs ground-truth pi_true, mean +/- std)",
              f"  {'PDE':<10}{'r_mech':>18}{'pi_true':>18}"]
        for pde in pdes:
            rm, rs = ms(pde, gname, 'r_mech', 'diagnostic')
            tm, ts = ms(pde, gname, 'pi_true', 'diagnostic')
            L.append(f"  {pde:<10}{rm:>+9.3f}+/-{rs:<6.3f}{tm:>+10.2e}+/-{ts:<7.2e}")
        L += ["", "GAUGE-DEPENDENT MECHANISM  (GENERIC, mean +/- std)",
              f"  {'PDE':<10}{'rho_M':>16}{'r_S':>16}{'r_E':>14}"]
        for pde in pdes:
            rm, rs = ms(pde, gname, 'rho_M', 'diagnostic')
            sm, ss = ms(pde, gname, 'r_S', 'diagnostic')
            em, es = ms(pde, gname, 'r_E', 'diagnostic')
            L.append(f"  {pde:<10}{rm:>8.3f}+/-{rs:<6.3f}{sm:>8.3f}+/-{ss:<6.3f}{em:>12.1e}")

    L += ["", "  LaTeX accuracy rows (PDE & Model & params & L2-1step & L2-roll):"]
    for pde in pdes:
        for m in models:
            p = runs[0][pde][m].get('params', 0)
            sm, ss = ms(pde, m, 'l2_single'); rm, rs = ms(pde, m, 'l2_rollout')
            L.append(f"  {pde.capitalize()} & {m} & {p/1e3:.0f}K & "
                     f"${sm:.3f}\\pm{ss:.3f}$ & ${rm:.3f}\\pm{rs:.3f}$ \\\\")
    if gname:
        L += ["", "  LaTeX gauge-invariant rows (PDE & r_mech & pi_true):"]
        for pde in pdes:
            rm, rs = ms(pde, gname, 'r_mech', 'diagnostic')
            tm, ts = ms(pde, gname, 'pi_true', 'diagnostic')
            L.append(f"  {pde.capitalize()} & ${rm:+.3f}\\pm{rs:.3f}$ & "
                     f"${tm:+.2e}\\pm{ts:.0e}$ \\\\")

    summary = "\n".join(L)
    print(summary)
    if save_dir:
        import os, pickle, time
        os.makedirs(save_dir, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'deeponet_1d_seeds_{ts}.pkl'), 'wb') as f:
            pickle.dump({'runs': runs, 'n_seeds': n_seeds}, f)
        with open(os.path.join(save_dir, f'deeponet_1d_seeds_{ts}.txt'), 'w') as f:
            f.write(summary)
        print(f"\nSaved seed-averaged summary + per-seed results -> {save_dir}")
    return runs


def run_in_colab():
    import os
    try:
        from google.colab import drive; drive.mount('/content/drive')
        sd = '/content/drive/MyDrive/GENERIC_FNO_results'
    except ImportError:
        sd = './GENERIC_FNO_results'
    os.makedirs(sd, exist_ok=True); print(f"Results -> {sd}\n")
    return run_benchmark(save_dir=sd)


if __name__ == '__main__':
    #run_in_colab()
    DRIVE = '/content/drive/MyDrive/GENERIC_FNO_results'

    # 1D FNO (~fast)
    run_benchmark_seeds(n_seeds=3, save_dir=DRIVE)


################################################################################
# SEED 0/4
################################################################################
GENERIC-DeepONet 1D STUDY (no E-supervision)  device=cuda
  degeneracy_construction=True min_diss_weight=0.0

PDE: HEAT

--- DeepONet ---
  Epoch   1: data=0.003058
  Epoch  50: rollout=0.000074 data=0.000027
  Epoch 100: data=0.000035 rollout=0.000032
  Epoch 150: rollout=0.000082 data=0.000032
  L2-1step=0.009558 rollout=0.053479 E-track=0.033794 Mono=55.83

--- GENERIC-DeepONet ---
  Epoch   1: data=0.000041 degeneracy=0.000000
  Epoch  50: rollout=0.000002 degeneracy=0.000000 data=0.000002
  Epoch 100: data=0.000000 degeneracy=0.000000 rollout=0.000001
  Epoch 150: rollout=0.000000 degeneracy=0.000000 data=0.000000
  L2-1step=0.001687 rollout=0.009233 E-track=0.003562 Mono=0.00
  r_E=3.87e-07 dE/step=2.21e-07 dS/step=1.38e-02
  saved checkpoint -> /content/drive/MyDrive/GENERIC_FNO_results/generic_deeponet_1d_h